In [3]:
!pip install albumentations segmentation-models-pytorch -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.7 MB/s eta 0:00:00


In [2]:
import os
import cv2
import numpy as np
from tqdm import tqdm

print("=== HỆ THỐNG XỬ LÝ DỮ LIỆU TỔNG HỢP (PHIÊN BẢN KAGGLE CHUẨN) ===")

TARGET_SIZE = (512, 512)

# 1. Định nghĩa đường dẫn theo chuẩn Kaggle
# Input: Nơi chứa dữ liệu gốc (Chỉ đọc)
# Working: Nơi lưu kết quả (Có quyền ghi)
KAGGLE_INPUT = '/kaggle/input/'
KAGGLE_WORKING = '/kaggle/working/'

combined_img_dir = os.path.join(KAGGLE_WORKING, 'dataset/train_combined/images/')
combined_mask_dir = os.path.join(KAGGLE_WORKING, 'dataset/train_combined/masks/')

os.makedirs(combined_img_dir, exist_ok=True)
os.makedirs(combined_mask_dir, exist_ok=True)

# Từ điển quy đổi
dg_color_to_id = {(255, 255, 0): 0, (0, 255, 255): 1, (255, 0, 255): 2, (0, 255, 0): 3, (255, 0, 0): 4, (255, 255, 255): 5}
loveda_to_dg = {1: 5, 2: 0, 3: 0, 4: 4, 5: 5, 6: 3, 7: 1}

# ==========================================
# PHẦN 1: XỬ LÝ DEEPGLOBE
# ==========================================
print("\n[1/2] Đang quét radar tìm DeepGlobe...")
dg_img_paths = []
for root, dirs, files in os.walk(KAGGLE_INPUT):
    if 'deepglobe' in root.lower() and 'train' in root.lower() and 'sample' not in root.lower():
        for file in files:
            if file.endswith('_sat.jpg'):
                dg_img_paths.append(os.path.join(root, file))

if not dg_img_paths:
    print("⚠️ Không tìm thấy ảnh DeepGlobe. Hãy kiểm tra mục Input bên phải!")
else:
    print(f"-> Tìm thấy {len(dg_img_paths)} ảnh DeepGlobe. Đang xử lý...")
    for img_path in tqdm(dg_img_paths):
        img_name = os.path.basename(img_path)
        mask_path = img_path.replace('_sat.jpg', '_mask.png')
        
        img = cv2.imread(img_path)
        mask_bgr = cv2.imread(mask_path)
        if img is None or mask_bgr is None: continue
            
        img_res = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        mask_res = cv2.resize(mask_bgr, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        
        gray_mask = np.zeros(TARGET_SIZE, dtype=np.uint8)
        for color, class_id in dg_color_to_id.items():
            gray_mask[np.all(mask_res == color, axis=-1)] = class_id
            
        cv2.imwrite(os.path.join(combined_img_dir, f"DG_{img_name}"), img_res)
        cv2.imwrite(os.path.join(combined_mask_dir, f"DG_{img_name.replace('_sat.jpg', '_mask.png')}"), gray_mask)

# ==========================================
# PHẦN 2: XỬ LÝ LOVEDA
# ==========================================
print("\n[2/2] Đang quét radar tìm LoveDA...")
loveda_paths = []
for root, dirs, files in os.walk(KAGGLE_INPUT):
    if 'loveda' in root.lower() and 'train' in root.lower() and 'images_png' in root.lower():
        for file in files:
            if file.endswith('.png'):
                loveda_paths.append(os.path.join(root, file))

if not loveda_paths:
    print("⚠️ Không tìm thấy ảnh LoveDA. Hãy kiểm tra mục Input bên phải!")
else:
    print(f"-> Tìm thấy {len(loveda_paths)} ảnh LoveDA. Đang xử lý...")
    for img_path in tqdm(loveda_paths):
        img_name = os.path.basename(img_path)
        mask_path = img_path.replace('images_png', 'masks_png')
        
        img = cv2.imread(img_path)
        mask_gray = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask_gray is None: continue
            
        img_res = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        mask_res = cv2.resize(mask_gray, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        
        new_mask = np.zeros_like(mask_res)
        for l_id, dg_id in loveda_to_dg.items():
            new_mask[mask_res == l_id] = dg_id
            
        cv2.imwrite(os.path.join(combined_img_dir, f"LDA_{img_name.replace('.png', '.jpg')}"), img_res)
        cv2.imwrite(os.path.join(combined_mask_dir, f"LDA_{img_name}"), new_mask)

print(f"\n✅ XỬ LÝ XONG! Tổng cộng có: {len(os.listdir(combined_img_dir))} ảnh trong kho lưu trữ.")

=== HỆ THỐNG XỬ LÝ DỮ LIỆU TỔNG HỢP (PHIÊN BẢN KAGGLE CHUẨN) ===

[1/2] Đang quét radar tìm DeepGlobe...
-> Tìm thấy 803 ảnh DeepGlobe. Đang xử lý...


100%|██████████| 803/803 [02:43<00:00,  4.92it/s]



[2/2] Đang quét radar tìm LoveDA...
-> Tìm thấy 2522 ảnh LoveDA. Đang xử lý...


100%|██████████| 2522/2522 [02:58<00:00, 14.14it/s]


✅ XỬ LÝ XONG! Tổng cộng có: 3325 ảnh trong kho lưu trữ.


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import segmentation_models_pytorch as smp
from tqdm import tqdm

# 1. THIẾT LẬP THAM SỐ (ĐÃ TỐI ƯU ĐỂ TRÁNH LỖI OOM)
DATA_DIR = '/kaggle/working/dataset/train_combined/'
IMG_DIR = os.path.join(DATA_DIR, 'images')
MASK_DIR = os.path.join(DATA_DIR, 'masks')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Giải phóng bộ nhớ GPU còn sót lại
import gc
torch.cuda.empty_cache()
gc.collect()

# GIẢM BATCH_SIZE ĐỂ KHÔNG BỊ TRÀN RAM GPU
BATCH_SIZE = 8# Nếu vẫn lỗi hãy giảm xuống 4, nhưng 8 thường là con số an toàn cho T4
EPOCHS = 40

# 2. LỚP NẠP DỮ LIỆU TỐI ƯU
class KaggleCombinedDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        # Lấy danh sách ảnh gốc (.jpg)
        self.images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # LOGIC TÌM MASK THÔNG MINH:
        if 'DG_' in img_name:
            # Nếu là DeepGlobe: Ảnh 'DG_123_sat.jpg' -> Mask 'DG_123_mask.png'
            mask_name = img_name.replace('_sat.jpg', '_mask.png')
        else:
            # Nếu là LoveDA: Ảnh 'LDA_123.jpg' -> Mask 'LDA_123.png'
            mask_name = img_name.replace('.jpg', '.png')
            
        mask_path = os.path.join(self.mask_dir, mask_name)

        # Đọc ảnh
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Đọc mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Kiểm tra an toàn để không bao giờ bị NoneType nữa
        if mask is None:
            # Nếu vẫn không thấy, thử tìm phương án dự phòng cuối cùng
            alt_mask_name = img_name.replace('.jpg', '.png')
            mask = cv2.imread(os.path.join(self.mask_dir, alt_mask_name), cv2.IMREAD_GRAYSCALE)
            
        if mask is None:
            raise FileNotFoundError(f"❌ Không tìm thấy mask cho ảnh: {img_name} tại {mask_path}")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()

# 3. TĂNG CƯỜNG DỮ LIỆU (AUGMENTATION)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Transpose(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

dataset = KaggleCombinedDataset(IMG_DIR, MASK_DIR, transform=train_transform)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

# 4. KHỞI TẠO MÔ HÌNH V3 MASTER
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet", # Sẽ tải từ internet nếu Internet ON
    in_channels=3,
    classes=6 
).to(DEVICE)

# Hàm Loss kết hợp giúp ranh giới các vùng đất sắc nét hơn
criterion_dice = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
criterion_ce = nn.CrossEntropyLoss()
def combo_loss(pred, target): return criterion_dice(pred, target) + criterion_ce(pred, target)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 5. TIẾN HÀNH RÈN LUYỆN
print(f"🚀 KÍCH HOẠT SIÊU MÁY TÍNH KAGGLE")
print(f"📚 Tổng số ảnh huấn luyện: {len(dataset)}")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    
    for images, masks in loop:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = combo_loss(outputs, masks)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    scheduler.step()

# LƯU THÀNH QUẢ CUỐI CÙNG
torch.save(model.state_dict(), "unet_resnet50_V3_Master.pth")
print("\n🎉 CHÚC MỪNG! ĐÃ LUYỆN XONG SIÊU MÔ HÌNH V3 MASTER.")

🚀 KÍCH HOẠT SIÊU MÁY TÍNH KAGGLE
📚 Tổng số ảnh huấn luyện: 3325


Epoch 2/40:  49%|████▉     | 205/416 [02:26<02:30,  1.40it/s, loss=1.25] 

In [8]:
import torch
import segmentation_models_pytorch as smp
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. Định nghĩa lại các thông số cơ bản
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "/kaggle/input/datasets/duyhuyynh/model-file/unet_resnet50_V3_Master.pth" # Đường dẫn file vừa train xong

# 2. Khai báo lại cấu trúc Model (Phải giống hệt lúc train)
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None, # Không cần imagenet vì ta sẽ load trọng số của ta
    in_channels=3,
    classes=6
).to(DEVICE)

# 3. Nạp trọng số từ file .pth vào model
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Đã nạp bộ não V3 Master thành công! Sẵn sàng Test.")
else:
    print("❌ Không tìm thấy file .pth. Bạn hãy kiểm tra tên file trong mục Output nhé!")

# 4. Khai báo lại transform để test
test_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

✅ Đã nạp bộ não V3 Master thành công! Sẵn sàng Test.


In [10]:
# Khai báo lại transform chuẩn cho Test
test_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Định nghĩa lại dataset test từ thư mục working
# Lưu ý: Ô này chỉ chạy được nếu bạn ĐÃ CHẠY Pipeline xử lý ảnh trước đó
IMG_DIR = '/kaggle/working/dataset/train_combined/images'
MASK_DIR = '/kaggle/working/dataset/train_combined/masks'

if os.path.exists(IMG_DIR):
    test_dataset = KaggleCombinedDataset(IMG_DIR, MASK_DIR, transform=test_transform)
    print(f"✅ Đã kết nối thành công với bộ dữ liệu Test! (Tổng số ảnh: {len(test_dataset)})")
else:
    print("❌ Lỗi: Không tìm thấy thư mục ảnh. Bạn có lỡ tay xóa /kaggle/working/ chưa?")

❌ Lỗi: Không tìm thấy thư mục ảnh. Bạn có lỡ tay xóa /kaggle/working/ chưa?
